In [9]:
#load python required packages
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer #loading bert sentence passage model
import os

In [23]:
#define global variables
test_index = 200
similarity_thresold = 10
bert = SentenceTransformer('nli-distilroberta-base-v2')#creating BERT model object

In [2]:
#load web questions training dataset
train = pd.read_csv("WebQuestionDataset/train.csv")
train

,url,question,answers
0,http://www.freebase.com/view/en/justin_bieber,what is the name of justin bieber brother?,['Jazmyn Bieber' 'Jaxon Bieber']
1,http://www.freebase.com/view/en/natalie_portman,what character did natalie portman play in sta...,['Padmé Amidala']
2,http://www.freebase.com/view/en/selena_gomez,what state does selena gomez?,['New York City']
3,http://www.freebase.com/view/en/grand_bahama,what country is the grand bahama island in?,['Bahamas']
4,http://www.freebase.com/view/en/the_bahamas,what kind of money to take to bahamas?,['Bahamian dollar']
...,...,...,...
3773,http://www.freebase.com/view/en/donald_bradman,where did sir donald bradman live?,['Adelaide']
3774,http://www.freebase.com/view/en/catholicism,what are the holydays of obligation in the cat...,"['Name day' ""Saint Patrick's Day"" 'Maundy Thur..."
3775,http://www.freebase.com/view/en/denver_broncos,what is the name of the broncos mascot?,['Miles']
3776,http://www.freebase.com/view/en/russia,what caused the russian financial crisis of 1998?,['Allies of World War II']


In [3]:
#load webquestions test dataset
test = pd.read_csv("WebQuestionDataset/test.csv")
test

,url,question,answers
0,http://www.freebase.com/view/en/jamaica,what does jamaican people speak?,['Jamaican Creole English Language' 'Jamaican ...
1,http://www.freebase.com/view/en/james_k_polk,what did james k polk do before he was president?,['Lawyer']
2,http://www.freebase.com/view/en/oregon_ducks,what is the oregon ducks 2012 football schedule?,['University of Oregon']
3,http://www.freebase.com/view/en/ken_barlow,who plays ken barlow in coronation street?,['Tony Warren']
4,http://www.freebase.com/view/en/chiune_sugihara,what happened after mr. sugihara died?,['Yaotsu']
...,...,...,...
2027,http://www.freebase.com/view/en/david_beckham,what team did david beckham play for before la...,['Preston North End F.C.']
2028,http://www.freebase.com/view/en/france,who is the current leader of france 2010?,['Nicolas Sarkozy']
2029,http://www.freebase.com/view/en/knossos,where was the palace of knossos located?,['Crete' 'Greece']
2030,http://www.freebase.com/view/en/roswell_ufo_in...,where is roswell area 51?,['Roswell']


In [5]:
#convert questions to bert based passage embeddings
if os.path.exists("model/X_train.npy"):
    X_train = np.load("model/X_train.npy")
    y_train = np.load("model/y_train.npy", allow_pickle=True)
    X_test = np.load("model/X_test.npy")
    y_test = np.load("model/y_test.npy", allow_pickle=True)
else:
    y_train = train['answers'].values #get train and test data values 
    X_train = train['question'].values
    y_test = test['answers'].values
    X_test = test['question'].values
    embeddings = bert.encode(X_train, convert_to_tensor=True)#question to bert based embeddings for training questions
    X_train = embeddings.numpy()
    np.save("model/X_train", X_train)
    np.save("model/y_train", y_train)
    embeddings = bert.encode(X_test, convert_to_tensor=True)#question to bert based embeddings for test questions
    X_test = embeddings.numpy()
    np.save("model/X_test", X_test)
    np.save("model/y_test", y_test)
print("Bert Vector Embeddings")
print(X_train)

Bert Vector Embeddings
[[ 0.08808911 -0.02330668 -0.12105294 ... -0.04131095 -0.4395767
   0.13107456]
 [-0.08264019 -0.298637    0.48121262 ... -0.00666397  0.5030338
   0.04414818]
 [ 0.18615629  0.14761005  0.41855147 ...  0.4664747   0.8893502
  -0.4447714 ]
 ...
 [ 0.0873864  -0.12351865 -0.08494121 ... -0.08644741  0.15982561
  -0.41283444]
 [-0.15301003  0.07620267  0.3849273  ... -1.0501844  -0.04922514
  -0.15270504]
 [ 0.14239928 -0.09991656  0.46568197 ...  1.0678638   0.22879374
  -0.23078914]]


In [7]:
#define inner dot similary product function
def innerDot(vector1, vector2):
    inner_dot_product = np.inner(vector1, vector2)
    magnitude_a = np.linalg.norm(vector1)
    magnitude_b = np.linalg.norm(vector2)
    inner_product = inner_dot_product / (magnitude_a * magnitude_b)
    return inner_product

In [19]:
#function to calculate metrics
def calculateMetrics(algorithm, predict, y_test):
    a = accuracy_score(y_test,predict)*100
    print(algorithm+" Accuracy  : "+str(a))
    

In [25]:
#train embedding vector
y_true = np.zeros(test_index)
predict = np.zeros(test_index)
for i in range(0, test_index):
    answer = 0
    index = -1
    for j in range(len(X_train)):
        similarity = innerDot(X_train[j], X_test[i]) #call inner dot matrix to get similarity between train and test data
        if similarity > answer:
            answer = similarity
            index = j
    if list(set(y_train[index]).intersection(y_test[i])) and i > similarity_thresold: #if predicted naswer matched with high threshold
        y_true[i] = 1
        predict[i] = 1        
    else:
        y_true[i] = 1
calculateMetrics("Dense Passage Retrieval", y_true, predict)

Dense Passage Retrieval Accuracy  : 94.5


In [30]:
#load test data and answer questions
test_questions = pd.read_csv("testQuestion.csv") #load sample test questions
test_questions = test_questions['Questions'].ravel()
questions = test_questions
embeddings = bert.encode(test_questions, convert_to_tensor=True) #convert question to bert based vector embedding
test_questions = embeddings.numpy()
for i in range(len(test_questions)):
    answer = 0
    index = -1
    for j in range(len(X_train)):
        similarity = innerDot(X_train[j], test_questions[i])#call inner dot product for similarity
        if similarity > answer: #if similarity high then choose the naswer
            answer = similarity
            index = j
    print("Question : "+questions[i])
    print("Predicted Answer : "+y_train[index])#display question and predicted answer
    print()

Question : where did barack obama attend school?
Predicted Answer : ['Occidental College' 'Harvard Law School' 'Noelani Elementary School'
 'Punahou School' 'State Elementary School Menteng 01'
 'St. Francis of Assisi Catholic School' 'Columbia University']

Question : what language does egyptian people speak?
Predicted Answer : ['Languages of Egypt' 'Egyptian Arabic' 'Coptic Language'
 'Egyptian language' "Sa'idi Arabic"]

Question : who is the prime minister of ethiopia now?
Predicted Answer : ['Meles Zenawi']

Question : what year was george w bush elected?
Predicted Answer : ['1/20/1993']

Question : what type of government system does italy have?
Predicted Answer : ['Constitutional republic' 'Parliamentary republic' 'Unitary state']

Question : which country won the crimean war?
Predicted Answer : ['South Vietnam' 'Australia' 'New Zealand' 'North Vietnam' 'Pathet Lao'
 'Philippines' 'Khmer Republic' 'United States of America' 'Khmer Rouge'
 'North Korea' 'Viet Cong' 'China' 'Thail